# Fraud Detection — Graph + XGBoost Pipeline

This is the **exact pipeline** used in this project (`fraud_graph.py`).

The core idea: instead of scoring each transaction in isolation, we first build an **Identity Graph** that connects transactions through shared identifiers (card, device, address, email). Historical fraud patterns on those shared identifiers become features for XGBoost.

**Steps:**
1. Import Libraries
2. Load IEEE-CIS Dataset
3. Temporal Split (History vs Score set)
4. Build Identity Graph from History
5. Extract Graph Features (no label leakage)
6. Encode Categorical Features
7. Train XGBoost with Graph Features
8. Evaluate the Model
9. Visualise — Feature Importance, ROC, PR, Score Distribution, Fraud Ring Graph
10. Save Model Artifacts
11. Demo — Score a New Transaction

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import json, os
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve, classification_report
)
from pyvis.network import Network
import joblib

print('All libraries imported successfully.')

---
## Step 2 — Load IEEE-CIS Dataset

We load two CSV files and merge them on `TransactionID`:
- `train_transaction.csv` — transaction-level features (amount, card, address, email, C/D/M columns)
- `train_identity.csv` — device-level features (DeviceType, DeviceInfo, id_* columns)

We only load the columns we actually need to keep memory usage low.

In [ ]:
TX_PATH  = '/home/hayakreevan/Downloads/ieee-fraud-detection/train_transaction.csv'
IDE_PATH = '/home/hayakreevan/Downloads/ieee-fraud-detection/train_identity.csv'
OUT_DIR  = '/home/hayakreevan/Downloads/Use_Case_Fraud_Detection'

tx_cols = [
    'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
    'ProductCD', 'card1', 'card4', 'card6',
    'addr1', 'P_emaildomain', 'R_emaildomain',
    'C1', 'C2', 'C5', 'C6', 'C13', 'C14',
    'D1', 'D4', 'D10', 'D15',
    'M1', 'M4', 'M6',
]
ide_cols = ['TransactionID', 'DeviceType', 'DeviceInfo', 'id_12', 'id_30', 'id_31']

tx  = pd.read_csv(TX_PATH,  usecols=tx_cols)
ide = pd.read_csv(IDE_PATH, usecols=ide_cols)

# merge and sort chronologically by TransactionDT
df = tx.merge(ide, on='TransactionID', how='left')
df = df.sort_values('TransactionDT').reset_index(drop=True)

print(f'Total rows : {len(df):,}')
print(f'Fraud rate : {df["isFraud"].mean()*100:.2f}%')
print(f'Fraud cases: {df["isFraud"].sum():,}')
df.head(3)

In [ ]:
# Fill null graph key columns with 'UNKNOWN'
for col in ['card1', 'addr1', 'P_emaildomain', 'DeviceInfo', 'DeviceType']:
    df[col] = df[col].fillna('UNKNOWN').astype(str)

# Class distribution
df['isFraud'].value_counts().plot(kind='bar', color=['steelblue','crimson'],
                                   title='Fraud vs Legit')
plt.xticks([0,1], ['Legit','Fraud'], rotation=0)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

---
## Step 3 — Temporal Split

**Why temporal and not random?**

Fraud detection is a time-series problem. If we randomly split, future transactions leak into the graph history. We must train only on the past and predict the future — exactly how production works.

- **History (70%)** → used to build the Identity Graph
- **Score set (30%)** → used to extract graph features and train/evaluate XGBoost

In [ ]:
split_idx  = int(len(df) * 0.70)
df_history = df.iloc[:split_idx].copy()
df_score   = df.iloc[split_idx:].copy()

print(f'History : {len(df_history):,} rows  (fraud {df_history["isFraud"].mean()*100:.2f}%)')
print(f'Score   : {len(df_score):,} rows  (fraud {df_score["isFraud"].mean()*100:.2f}%)')

---
## Step 4 — Build Identity Graph from History

We build a **bipartite graph** using NetworkX:

```
Transaction ──USES_CARD──► Card
Transaction ──BILLED_TO──► Address
Transaction ──EMAIL_DOMAIN► Email
Transaction ──FROM_DEVICE──► Device
```

**Hub nodes** (Card, Address, Email, Device) store historical fraud counts. If many fraud transactions share the same card or device, that hub becomes a fraud signal for any future transaction using it.

In [ ]:
G = nx.Graph()

def compute_hub_stats(df, col, prefix):
    """Returns dict: prefixed_hub_id → {total_tx, fraud_tx}"""
    stats = df.groupby(col)['isFraud'].agg(['count','sum']).reset_index()
    stats.columns = [col, 'total_tx', 'fraud_tx']
    return {
        f"{prefix}{row[col]}": {'total_tx': int(row['total_tx']), 'fraud_tx': int(row['fraud_tx'])}
        for _, row in stats.iterrows()
    }

print('Computing hub stats...')
card_stats   = compute_hub_stats(df_history, 'card1',         'CARD_')
addr_stats   = compute_hub_stats(df_history, 'addr1',         'ADDR_')
email_stats  = compute_hub_stats(df_history, 'P_emaildomain', 'EMAIL_')
device_stats = compute_hub_stats(df_history, 'DeviceInfo',    'DEV_')

# Add hub nodes
for nid, attrs in card_stats.items():   G.add_node(nid, node_type='card',    **attrs)
for nid, attrs in addr_stats.items():   G.add_node(nid, node_type='address', **attrs)
for nid, attrs in email_stats.items():  G.add_node(nid, node_type='email',   **attrs)
for nid, attrs in device_stats.items(): G.add_node(nid, node_type='device',  **attrs)

# Add transaction nodes + edges
print('Adding transaction nodes and edges (this may take a minute)...')
for _, row in df_history.iterrows():
    tid = f"TX_{row['TransactionID']}"
    G.add_node(tid, node_type='transaction', fraud=int(row['isFraud']), amount=row['TransactionAmt'])
    G.add_edge(tid, f"CARD_{row['card1']}",         edge_type='USES_CARD')
    G.add_edge(tid, f"ADDR_{row['addr1']}",         edge_type='BILLED_TO')
    G.add_edge(tid, f"EMAIL_{row['P_emaildomain']}",edge_type='EMAIL_DOMAIN')
    G.add_edge(tid, f"DEV_{row['DeviceInfo']}",     edge_type='FROM_DEVICE')

n_types = {}
for n, d in G.nodes(data=True):
    t = d.get('node_type','?')
    n_types[t] = n_types.get(t,0) + 1

print(f'\nGraph built:')
print(f'  Nodes : {G.number_of_nodes():,}')
print(f'  Edges : {G.number_of_edges():,}')
for t, c in sorted(n_types.items()):
    print(f'  {t:<15}: {c:,}')

In [ ]:
# Save the graph as GraphML
os.makedirs(f'{OUT_DIR}/model', exist_ok=True)
nx.write_graphml(G, f'{OUT_DIR}/model/identity_graph.graphml')
print(f'Graph saved → model/identity_graph.graphml  ({G.number_of_nodes():,} nodes)')

---
## Step 5 — Extract Graph Features (no label leakage)

For every transaction in the **score set**, we look up its card/device/address/email in the history graph and extract:

| Feature | What it means |
|---|---|
| `card_fraud_rate` | % of historical fraud txns on this card |
| `card_tx_volume` | total txns seen on this card in history |
| `card_ring_size` | how many txns share this card (ring size) |
| `device_fraud_rate` | % of historical fraud txns on this device |
| `device_ring_size` | how many txns share this device |
| `addr_fraud_rate` | fraud rate at this billing address |
| `email_fraud_rate` | fraud rate for this email domain |
| `card_seen_in_hist` | was this card seen in history at all? |
| `device_seen_in_hist` | was this device seen in history at all? |

All lookups come from history only — no future data leaks in.

In [ ]:
def hub_lookup(stats_dict, prefix):
    frate = {k.replace(prefix,''): v['fraud_tx']/max(v['total_tx'],1) for k,v in stats_dict.items()}
    total = {k.replace(prefix,''): v['total_tx'] for k,v in stats_dict.items()}
    rsize = {}
    for nid in stats_dict:
        rsize[nid.replace(prefix,'')] = sum(
            1 for nb in G.neighbors(nid)
            if G.nodes[nb].get('node_type') == 'transaction')
    return frate, total, rsize

card_frate,  card_total,  card_rsize  = hub_lookup(card_stats,   'CARD_')
addr_frate,  addr_total,  _           = hub_lookup(addr_stats,   'ADDR_')
dev_frate,   dev_total,   dev_rsize   = hub_lookup(device_stats, 'DEV_')
email_frate, _,           _           = hub_lookup(email_stats,  'EMAIL_')

feat_df = df_score[[
    'TransactionAmt','ProductCD','card4','card6','DeviceType','P_emaildomain',
    'C1','C2','C5','C6','C13','C14','D1','D4','D10','isFraud',
    'card1','addr1','DeviceInfo',
]].copy().rename(columns={'isFraud': 'fraud_label'})

feat_df['card_fraud_rate']     = feat_df['card1'].map(card_frate).fillna(0)
feat_df['card_tx_volume']      = feat_df['card1'].map(card_total).fillna(0)
feat_df['card_ring_size']      = feat_df['card1'].map(card_rsize).fillna(0)
feat_df['addr_fraud_rate']     = feat_df['addr1'].map(addr_frate).fillna(0)
feat_df['addr_tx_volume']      = feat_df['addr1'].map(addr_total).fillna(0)
feat_df['device_fraud_rate']   = feat_df['DeviceInfo'].map(dev_frate).fillna(0)
feat_df['device_tx_volume']    = feat_df['DeviceInfo'].map(dev_total).fillna(0)
feat_df['device_ring_size']    = feat_df['DeviceInfo'].map(dev_rsize).fillna(0)
feat_df['email_fraud_rate']    = feat_df['P_emaildomain'].map(email_frate).fillna(0)
feat_df['card_seen_in_hist']   = feat_df['card1'].isin(card_frate).astype(int)
feat_df['device_seen_in_hist'] = feat_df['DeviceInfo'].isin(dev_frate).astype(int)
feat_df = feat_df.drop(columns=['card1','addr1','DeviceInfo'])

print(f'Feature matrix : {feat_df.shape}')
print(f'Fraud cases    : {feat_df["fraud_label"].sum():,} ({feat_df["fraud_label"].mean()*100:.2f}%)')

# Graph feature means by label — shows graph features separate fraud from legit
gf = ['card_fraud_rate','device_fraud_rate','addr_fraud_rate','card_ring_size','device_ring_size']
print('\nGraph feature means by fraud label:')
print(feat_df[gf + ['fraud_label']].groupby('fraud_label').mean().round(4))

---
## Step 6 — Encode Categorical Features

XGBoost needs numeric inputs. We LabelEncode categorical columns using the full dataset so the encoder covers all possible values (including those only in history or score set).

In [ ]:
cat_cols = ['ProductCD', 'card4', 'card6', 'DeviceType', 'P_emaildomain']

for col in cat_cols:
    le = LabelEncoder()
    le.fit(df[col].fillna('UNKNOWN').astype(str))
    feat_df[col] = le.transform(
        feat_df[col].astype(str).map(
            lambda x, le=le: x if x in le.classes_ else le.classes_[0]
        )
    )

feat_df = feat_df.fillna(0)
print('Encoding done. Sample:')
feat_df[cat_cols].head(3)

---
## Step 7 — Train XGBoost with Graph Features

We use **XGBoost** because it handles mixed feature types well and `scale_pos_weight` addresses class imbalance without needing SMOTE.

Key hyperparameters:
- `n_estimators=500` — 500 boosting rounds
- `max_depth=6` — moderate depth to avoid overfitting
- `scale_pos_weight` — automatically set to `legit_count / fraud_count`
- `early_stopping_rounds=30` — stops if AUC doesn't improve for 30 rounds

In [ ]:
FEATURE_COLS = [
    'TransactionAmt', 'ProductCD', 'card4', 'card6', 'DeviceType',
    'C1', 'C2', 'C5', 'C6', 'C13', 'C14', 'D1', 'D4', 'D10',
    # graph features
    'card_fraud_rate', 'card_tx_volume', 'card_ring_size',
    'addr_fraud_rate', 'addr_tx_volume',
    'device_fraud_rate', 'device_tx_volume', 'device_ring_size',
    'email_fraud_rate',
    'card_seen_in_hist', 'device_seen_in_hist',
]

GRAPH_FEATURES = [
    'card_fraud_rate', 'card_tx_volume', 'card_ring_size',
    'addr_fraud_rate', 'addr_tx_volume',
    'device_fraud_rate', 'device_tx_volume', 'device_ring_size',
    'email_fraud_rate', 'card_seen_in_hist', 'device_seen_in_hist',
]

X = feat_df[FEATURE_COLS]
y = feat_df['fraud_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'Train size        : {len(X_train):,}  (fraud: {y_train.sum():,})')
print(f'Test size         : {len(X_test):,}   (fraud: {y_test.sum():,})')
print(f'scale_pos_weight  : {scale_pos:.1f}')

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
    verbosity=0,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('\nTraining complete.')

---
## Step 8 — Evaluate the Model

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred  = (y_proba >= 0.5).astype(int)

auc    = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print(f'ROC-AUC        : {auc:.4f}')
print(f'PR-AUC         : {pr_auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Legit','Fraud']))

---
## Step 9 — Visualisations

In [ ]:
# ── Confusion Matrix + ROC + PR Curve
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Legit','Fraud'],
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix  (AUC={auc:.3f})')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, '#e74c3c', lw=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve')
axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(rec, prec, '#2980b9', lw=2, label=f'PR AUC = {pr_auc:.4f}')
axes[2].axhline(y_test.mean(), color='k', linestyle='--', lw=1,
                label=f'Baseline (fraud rate={y_test.mean():.3f})')
axes[2].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/roc_pr_curves.png', dpi=150)
plt.show()
print('Saved: roc_pr_curves.png')

In [ ]:
# ── Feature Importance (graph features in red, transaction features in blue)
importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

colors = ['#e74c3c' if f in GRAPH_FEATURES else '#3498db' for f in importance['feature']]

fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(importance['feature'], importance['importance'], color=colors)
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance\nGraph features (red) vs Transaction features (blue)')
ax.invert_yaxis()
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Graph features (historical)'),
    mpatches.Patch(color='#3498db', label='Transaction features'),
], loc='lower right')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/feature_importance.png', dpi=150)
plt.show()
print('Saved: feature_importance.png')
print(importance.to_string(index=False))

In [ ]:
# ── Score Distribution
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_proba[y_test == 0], bins=60, alpha=0.6, color='#2ecc71', label='Legit', density=True)
ax.hist(y_proba[y_test == 1], bins=60, alpha=0.8, color='#e74c3c', label='Fraud', density=True)
ax.axvline(0.5, color='black', linestyle='--', label='Threshold 0.5')
ax.set(xlabel='Fraud Score', ylabel='Density', title='Fraud Score Distribution')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/score_distribution.png', dpi=150)
plt.show()
print('Saved: score_distribution.png')

In [ ]:
# ── Fraud Ring Graph (static PNG)
# Pick top fraud cards and devices as seed nodes
fraud_cards = sorted(
    [n for n,d in G.nodes(data=True) if d.get('node_type')=='card' and d.get('fraud_tx',0)>=2],
    key=lambda n: G.nodes[n]['fraud_tx'], reverse=True)[:12]

fraud_devices = sorted(
    [n for n,d in G.nodes(data=True) if d.get('node_type')=='device' and d.get('fraud_tx',0)>=2],
    key=lambda n: G.nodes[n]['fraud_tx'], reverse=True)[:6]

seed_nodes = set(fraud_cards + fraud_devices)
ego_nodes  = set(seed_nodes)
for s in seed_nodes:
    for nb in G.neighbors(s):
        if G.nodes[nb].get('node_type') == 'transaction':
            ego_nodes.add(nb)
            for nb2 in G.neighbors(nb):
                if G.nodes[nb2].get('node_type') in ('card','device','address'):
                    ego_nodes.add(nb2)
sub = G.subgraph(list(ego_nodes)[:200])

color_map = {'card':'#3498db','address':'#f39c12','device':'#9b59b6','email':'#1abc9c'}
node_colors, node_sizes = [], []
for n in sub.nodes():
    nd = G.nodes[n]
    nt = nd.get('node_type','transaction')
    if nt == 'transaction':
        node_colors.append('#e74c3c' if nd.get('fraud',0) else '#2ecc71')
        node_sizes.append(60)
    else:
        node_colors.append(color_map.get(nt,'#aaa'))
        node_sizes.append(220)

pos = nx.spring_layout(sub, seed=42, k=0.6)
fig, ax = plt.subplots(figsize=(16,12))
nx.draw_networkx_nodes(sub, pos, node_color=node_colors, node_size=node_sizes, ax=ax, alpha=0.88)
nx.draw_networkx_edges(sub, pos, alpha=0.2, ax=ax, edge_color='#888')
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Fraud Transaction'),
    mpatches.Patch(color='#2ecc71', label='Legit Transaction'),
    mpatches.Patch(color='#3498db', label='Card (hub)'),
    mpatches.Patch(color='#9b59b6', label='Device (hub)'),
    mpatches.Patch(color='#f39c12', label='Address (hub)'),
], loc='upper left', fontsize=10)
ax.set_title(
    f'Identity Fraud Ring Graph — {sub.number_of_nodes()} nodes, {sub.number_of_edges()} edges\n'
    'Hub nodes (cards/devices) shared across fraud + legit transactions reveal rings',
    fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fraud_ring_graph.png', dpi=150)
plt.show()
print('Saved: fraud_ring_graph.png')

In [ ]:
# ── Interactive HTML graph (open in browser)
net = Network(height='750px', width='100%', bgcolor='#1a1a2e', font_color='white', directed=False)
net.barnes_hut(gravity=-6000, central_gravity=0.3, spring_length=100)

for n in sub.nodes():
    nd = G.nodes[n]
    nt = nd.get('node_type','transaction')
    if nt == 'transaction':
        is_f  = nd.get('fraud',0)
        color = '#e74c3c' if is_f else '#2ecc71'
        title = f"<b>Transaction {n}</b><br>Amount: {nd.get('amount','?')}<br><b>Fraud: {bool(is_f)}</b>"
        net.add_node(n, color=color, shape='dot', title=title, size=8)
    elif nt == 'card':
        fr    = nd.get('fraud_tx',0) / max(nd.get('total_tx',1),1)
        color = '#e74c3c' if fr > 0.3 else '#3498db'
        title = f"<b>Card {n}</b><br>Total tx: {nd.get('total_tx','?')}<br>Fraud tx: {nd.get('fraud_tx','?')}<br>Fraud rate: {fr:.2%}"
        net.add_node(n, color=color, shape='square', title=title, size=18)
    elif nt == 'device':
        fr    = nd.get('fraud_tx',0) / max(nd.get('total_tx',1),1)
        title = f"<b>Device {n}</b><br>Fraud rate: {fr:.2%}"
        net.add_node(n, color='#9b59b6', shape='triangle', title=title, size=18)
    else:
        fr    = nd.get('fraud_tx',0) / max(nd.get('total_tx',1),1)
        title = f"<b>{nt.title()} {n}</b><br>Fraud rate: {fr:.2%}"
        net.add_node(n, color=color_map.get(nt,'#aaa'), shape='diamond', title=title, size=14)

for u, v, ed in sub.edges(data=True):
    net.add_edge(u, v, color='#333333', title=ed.get('edge_type',''))

net.show_buttons(filter_=['physics'])
net.save_graph(f'{OUT_DIR}/fraud_graph_interactive.html')
print(f'Saved: fraud_graph_interactive.html  ← open in browser')

---
## Step 10 — Save Model Artifacts

Saved files used by the backend API:
- `model/fraud_xgb.json` — XGBoost native format (portable, version-safe)
- `model/fraud_xgb.pkl` — sklearn-compatible pickle
- `model/metadata.json` — feature list, AUC scores, model version

In [ ]:
os.makedirs(f'{OUT_DIR}/model', exist_ok=True)

model.save_model(f'{OUT_DIR}/model/fraud_xgb.json')
joblib.dump(model, f'{OUT_DIR}/model/fraud_xgb.pkl')

metadata = {
    'model_version'  : 'xgb-graph-v1.0',
    'model_type'     : 'XGBClassifier',
    'auc_roc'        : round(auc, 4),
    'avg_precision'  : round(pr_auc, 4),
    'feature_cols'   : FEATURE_COLS,
    'graph_features' : GRAPH_FEATURES,
    'n_features'     : len(FEATURE_COLS),
    'train_rows'     : len(X_train),
    'train_fraud'    : int(y_train.sum()),
    'threshold'      : 0.5,
    'dataset'        : 'ieee-cis-fraud-detection',
    'graph_nodes'    : G.number_of_nodes(),
    'graph_edges'    : G.number_of_edges(),
}
with open(f'{OUT_DIR}/model/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved: model/fraud_xgb.json')
print('Saved: model/fraud_xgb.pkl')
print('Saved: model/metadata.json')
print()
print(json.dumps(metadata, indent=2))

---
## Step 11 — Demo: Score a New Transaction

This is exactly how the backend API scores live transactions — it queries the graph for the card/device/address history features, combines them with the raw transaction fields, and runs them through XGBoost.

In [ ]:
def score_transaction(tx: dict) -> dict:
    row = pd.DataFrame([{
        'TransactionAmt'    : tx.get('amount', 100),
        'ProductCD'         : 2,
        'card4'             : 2,
        'card6'             : 0,
        'DeviceType'        : 1 if tx.get('device_type') == 'mobile' else 0,
        'C1': tx.get('C1',1), 'C2': tx.get('C2',1),
        'C5': tx.get('C5',0), 'C6': tx.get('C6',1),
        'C13': tx.get('C13',1), 'C14': tx.get('C14',1),
        'D1': tx.get('D1',100), 'D4': tx.get('D4',0), 'D10': tx.get('D10',0),
        # graph features from live graph lookup
        'card_fraud_rate'    : tx.get('card_fraud_rate',   0.0),
        'card_tx_volume'     : tx.get('card_tx_volume',    0),
        'card_ring_size'     : tx.get('card_ring_size',    0),
        'addr_fraud_rate'    : tx.get('addr_fraud_rate',   0.0),
        'addr_tx_volume'     : tx.get('addr_tx_volume',    0),
        'device_fraud_rate'  : tx.get('device_fraud_rate', 0.0),
        'device_tx_volume'   : tx.get('device_tx_volume',  0),
        'device_ring_size'   : tx.get('device_ring_size',  0),
        'email_fraud_rate'   : tx.get('email_fraud_rate',  0.0),
        'card_seen_in_hist'  : int(tx.get('card_seen_in_hist',   False)),
        'device_seen_in_hist': int(tx.get('device_seen_in_hist', False)),
    }])
    score = model.predict_proba(row)[0][1]
    risk  = [f for f in importance.head(8)['feature']
             if f in GRAPH_FEATURES and row[f].values[0] > 0]
    return {
        'fraud_score'   : round(float(score), 4),
        'synthetic_flag': score >= 0.5,
        'decision'      : 'DECLINE' if score >= 0.5 else 'APPROVE',
        'risk_factors'  : risk[:3],
    }

# Suspicious transaction — card has 85% historical fraud rate, shared device
fraud_tx = {
    'amount': 350, 'device_type': 'mobile',
    'C1':1,'C2':5,'C5':1,'C6':1,'C13':10,'C14':1,'D1':0,
    'card_fraud_rate':0.85, 'card_tx_volume':12, 'card_ring_size':12,
    'device_fraud_rate':0.70, 'device_tx_volume':8, 'device_ring_size':8,
    'addr_fraud_rate':0.40, 'addr_tx_volume':5, 'email_fraud_rate':0.60,
    'card_seen_in_hist':True, 'device_seen_in_hist':True,
}

# Clean transaction — known-good card, no fraud history
clean_tx = {
    'amount': 80, 'device_type': 'desktop',
    'C1':1,'C2':1,'C5':0,'C6':1,'C13':1,'C14':1,'D1':200,
    'card_fraud_rate':0.0, 'card_tx_volume':3, 'card_ring_size':3,
    'device_fraud_rate':0.0, 'device_tx_volume':2, 'device_ring_size':2,
    'addr_fraud_rate':0.0, 'addr_tx_volume':2, 'email_fraud_rate':0.0,
    'card_seen_in_hist':True, 'device_seen_in_hist':True,
}

r1 = score_transaction(fraud_tx)
r2 = score_transaction(clean_tx)

print('[SUSPICIOUS] Card with 85% historical fraud rate:')
print(f'  Fraud score  : {r1["fraud_score"]}')
print(f'  Decision     : {r1["decision"]}')
print(f'  Risk factors : {r1["risk_factors"]}')

print()
print('[CLEAN] Known-good card, no fraud history:')
print(f'  Fraud score  : {r2["fraud_score"]}')
print(f'  Decision     : {r2["decision"]}')
print(f'  Risk factors : {r2["risk_factors"]}')

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported libraries (NetworkX, XGBoost, pyvis, sklearn) |
| 2 | Loaded IEEE-CIS transaction + identity CSVs, merged on TransactionID |
| 3 | Temporal 70/30 split — history to build graph, score set to train model |
| 4 | Built Identity Graph: Transaction→Card, Address, Email, Device nodes with historical fraud counts |
| 5 | Extracted graph features per transaction (card_fraud_rate, ring_size, device_fraud_rate, etc.) with zero label leakage |
| 6 | LabelEncoded categorical columns using full dataset vocabulary |
| 7 | Trained XGBoost (500 trees, scale_pos_weight, early stopping) on graph + transaction features |
| 8 | Evaluated with ROC-AUC, PR-AUC, confusion matrix, classification report |
| 9 | Saved visualisations: feature importance, ROC/PR curves, score distribution, fraud ring graph (static + interactive HTML) |
| 10 | Saved model artifacts: fraud_xgb.json, fraud_xgb.pkl, metadata.json |
| 11 | Demo: scored a suspicious vs clean transaction using the trained model |

**Running this notebook end-to-end will reproduce the exact same model as `fraud_graph.py`.**